System Path Setup

In [1]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


Import Core Libraries and GEE Initialization 

In [2]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee
import pandas as pd
import geopandas as gpd
import folium # For interactive mapping
from pygbif import occurrences # For GBIF data
from configs.regions import kenyan_coast_roi # Your ROI

# Initialize GEE (essential for all GEE operations)
ee.Initialize(project='gaias-ark') # <--- REPLACE 'gaias-ark' with YOUR GEE Project ID

print("All core libraries imported and GEE initialized.")

Region of Interest for Kenyan Coast defined.
All core libraries imported and GEE initialized.


Acquire Elevation Data (NASA NASADEM)

In [4]:
# Cell 3: Acquire Elevation Data (NASA NASADEM)
print("--- Acquiring Elevation Data ---")

# NASA NASADEM Global Elevation Model (30m resolution)
nasadem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')

# Clip to ROI
elevation_roi = nasadem.clip(kenyan_coast_roi)

print("NASADEM Elevation data acquired and clipped.")
# print(f"Elevation image info (snippet): {str(elevation_roi.getInfo())[:200]}...")

# Visualize Elevation
elevation_vis_params = {
    'min': 0, 'max': 100, # Adjust min/max for typical coastal elevation in meters
    'palette': ['#006633', '#E5FFCC', '#F7E9AF', '#6054EE', '#60EEAA'] # Green to blue/purple
}

m_elevation = folium.Map(location=[kenyan_coast_roi.centroid().getInfo()['coordinates'][1],
                                  kenyan_coast_roi.centroid().getInfo()['coordinates'][0]],
                         zoom_start=8, tiles='OpenStreetMap')
folium.GeoJson(
    kenyan_coast_roi.getInfo(),
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.1}
).add_to(m_elevation)

map_id_dict_elevation = elevation_roi.getMapId(elevation_vis_params)
folium.TileLayer(
    tiles=map_id_dict_elevation['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='Elevation (m)'
).add_to(m_elevation)

folium.LayerControl().add_to(m_elevation)
m_elevation

--- Acquiring Elevation Data ---
NASADEM Elevation data acquired and clipped.


Acquire Climate Data (WorldClim - Precipitation & Temperature)

In [6]:
# Cell 4: Acquire Climate Data (WorldClim - Precipitation & Temperature)
print("--- Acquiring Climate Data (WorldClim) ---")

# WorldClim 2.1 Climate data (10 minute resolution, monthly average)
# This is an ImageCollection of 12 images (one for each month)
# We will take the mean/sum over the year for simplicity in this MVP.
worldclim_dataset = ee.ImageCollection("WORLDCLIM/V1/MONTHLY") # V1 or V2 - let's try V1 for direct access

# Filter to a specific year if available, or use a general average if WorldClim is static.
# WorldClim V1/MONTHLY is generally a historical average (1970-2000).
# It contains bands like 'tavg' (average temperature), 'prec' (precipitation).

# Select temperature (tavg) and precipitation (prec) bands
tavg_band = 'tavg' # Average temperature in degrees Celsius * 10 (e.g., 250 = 25C)
prec_band = 'prec' # Precipitation in mm

# Calculate annual mean temperature and total annual precipitation
annual_mean_temp = worldclim_dataset.select(tavg_band).mean().divide(10).rename('mean_annual_temp_C') # Convert to Celsius
annual_total_prec = worldclim_dataset.select(prec_band).sum().rename('total_annual_prec_mm')

# Clip to ROI
mean_temp_roi = annual_mean_temp.clip(kenyan_coast_roi)
total_prec_roi = annual_total_prec.clip(kenyan_coast_roi)

print("WorldClim Climate data (mean annual temp, total annual precip) acquired and clipped.")

# Visualize Mean Annual Temperature
temp_vis_params = {
    'min': 15, 'max': 35, # Expected temperature range for tropical coast
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red']
}

m_temp = folium.Map(location=[kenyan_coast_roi.centroid().getInfo()['coordinates'][1],
                             kenyan_coast_roi.centroid().getInfo()['coordinates'][0]],
                    zoom_start=8, tiles='OpenStreetMap')
folium.GeoJson(kenyan_coast_roi.getInfo(), name='Kenyan Coastal ROI', style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.6}).add_to(m_temp)
map_id_dict_temp = mean_temp_roi.getMapId(temp_vis_params)
folium.TileLayer(tiles=map_id_dict_temp['tile_fetcher'].url_format, attr='Google Earth Engine', overlay=True, name='Mean Annual Temp (C)').add_to(m_temp)
folium.LayerControl().add_to(m_temp)
m_temp

# Visualize Total Annual Precipitation
prec_vis_params = {
    'min': 500, 'max': 3000, # Expected precipitation range for tropical coast in mm
    'palette': ['#f7fcf0', '#e0f3db', '#ccebc5', '#a8ddb5', '#7bccc4', '#4eb3d3', '#2b8cbe', '#0868ac', '#084081'] # Light to dark blue
}

m_prec = folium.Map(location=[kenyan_coast_roi.centroid().getInfo()['coordinates'][1],
                             kenyan_coast_roi.centroid().getInfo()['coordinates'][0]],
                    zoom_start=8, tiles='OpenStreetMap')
folium.GeoJson(kenyan_coast_roi.getInfo(), name='Kenyan Coastal ROI', style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.1}).add_to(m_prec)
map_id_dict_prec = total_prec_roi.getMapId(prec_vis_params)
folium.TileLayer(tiles=map_id_dict_prec['tile_fetcher'].url_format, attr='Google Earth Engine', overlay=True, name='Total Annual Precip (mm)').add_to(m_prec)
folium.LayerControl().add_to(m_prec)
m_prec

--- Acquiring Climate Data (WorldClim) ---
WorldClim Climate data (mean annual temp, total annual precip) acquired and clipped.
